# 01 — Data Discovery: In-Vitro Regeneration Intelligence

**Case Study A — Plant Intelligence Lab**

This notebook establishes the empirical foundation for genomic prediction of *Arabidopsis thaliana* shoot regeneration. It does not fit the predictive models yet. Its purpose is to determine exactly what can be modelled from the public evidence.

The target biological question is:

\[
\widehat{Y}_{21d}=f(G,P,X_{15d})
\]

compared with the genomic-treatment baseline

\[
\widehat{Y}_{21d}=f(G,P).
\]

Before either model is credible, the phenotype definitions, accession identities, protocol structure, replicate structure, missingness, and genotype–phenotype overlap must be established.


## Public source

AraPheno Study 80:

**Genetic dissection of shoot regeneration from root explants in Arabidopsis (Lardon et al., 2020)**

- Study DOI: `10.21958/study:80`
- Original paper DOI: `10.1038/s42003-020-01274-9`
- Study design: 170 natural *Arabidopsis thaliana* accessions subjected to two shoot-regeneration protocol variants.
- Primary forecasting traits used here: regenerated shoot counts at Day 15 and Day 21 under protocols A and B.

The notebook retrieves data directly from the public AraPheno REST API. No synthetic biological observations are used.


In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw" / "arapheno" / "study_80"
INTERIM = ROOT / "data" / "interim" / "case_study_a"
PROCESSED = ROOT / "data" / "processed" / "case_study_a"
RESULTS = ROOT / "reports" / "results"
FIGURES = ROOT / "reports" / "figures"

for path in [RAW, INTERIM, PROCESSED, RESULTS, FIGURES]:
    path.mkdir(parents=True, exist_ok=True)

ARAPHENO = "https://arapheno.1001genomes.org/rest"
STUDY_ID = 80


## 1. Verify the phenotype catalogue

The notebook first queries the study catalogue rather than hard-coding assumptions about available traits. This protects the analysis against silent changes in naming and ensures that the chosen endpoints are traceable to AraPheno phenotype identifiers.


In [ ]:
def get_json(url: str, timeout: int = 60):
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()

study_phenotypes = get_json(f"{ARAPHENO}/study/{STUDY_ID}/phenotypes.json")

# AraPheno responses may be either a list or wrapped in a dictionary.
if isinstance(study_phenotypes, dict):
    for key in ("phenotypes", "data", "results"):
        if key in study_phenotypes and isinstance(study_phenotypes[key], list):
            study_phenotypes = study_phenotypes[key]
            break

catalogue = pd.json_normalize(study_phenotypes)
catalogue.head()


In [ ]:
# Locate the four shoot-count endpoints that support the Day-15 -> Day-21 forecasting design.
name_col = next(c for c in catalogue.columns if c.lower() in {"name", "phenotype_name"})
id_col = next(c for c in catalogue.columns if c.lower() in {"id", "phenotype_id"})

shoot_catalogue = (
    catalogue[catalogue[name_col].str.contains(r"^shoots\s+(15d|21d)\s+protocol\s+[ab]$", case=False, regex=True, na=False)]
    [[id_col, name_col]]
    .copy()
    .sort_values(name_col)
)

shoot_catalogue


The public AraPheno pages currently identify the four focal phenotypes as:

| Endpoint | AraPheno phenotype ID |
|---|---:|
| shoots 15d protocol a | 1267 |
| shoots 15d protocol b | 1274 |
| shoots 21d protocol a | 1281 |
| shoots 21d protocol b | 1288 |

The live catalogue query above must agree with these identifiers before analysis proceeds.


In [ ]:
EXPECTED = {
    "shoots 15d protocol a": 1267,
    "shoots 15d protocol b": 1274,
    "shoots 21d protocol a": 1281,
    "shoots 21d protocol b": 1288,
}

observed = {
    str(row[name_col]).lower(): int(row[id_col])
    for _, row in shoot_catalogue.iterrows()
}

assert observed == EXPECTED, f"Phenotype catalogue changed or did not match expected IDs. Observed: {observed}"
print("Verified focal phenotype IDs:", observed)


## 2. Retrieve raw phenotype records

Each endpoint is downloaded independently and stored exactly as returned by the public API. Raw files are treated as immutable provenance artifacts. Transformations occur only in `data/interim/` and `data/processed/`.


In [ ]:
retrieved_at = datetime.now(timezone.utc).isoformat()

raw_payloads = {}
for phenotype_name, phenotype_id in EXPECTED.items():
    url = f"{ARAPHENO}/phenotype/{phenotype_id}/values.json"
    payload = get_json(url)
    raw_payloads[phenotype_id] = payload

    raw_path = RAW / f"phenotype_{phenotype_id}.json"
    raw_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Retrieved {len(raw_payloads)} focal phenotype payloads at {retrieved_at}")


## 3. Normalize accessions and replicate-level observations

AraPheno phenotype pages report many more values than the 170 natural accessions because multiple explants/replicates are measured. The statistical unit therefore has to be made explicit.

This notebook preserves the raw replicate-level observations and also constructs accession-level summaries. Model 1 will use an aggregation rule only after the replicate structure has been inspected.


In [ ]:
def normalize_values(payload, phenotype_id: int, phenotype_name: str) -> pd.DataFrame:
    data = payload
    if isinstance(payload, dict):
        for key in ("values", "data", "results"):
            if key in payload and isinstance(payload[key], list):
                data = payload[key]
                break
    if not isinstance(data, list):
        raise TypeError(f"Unsupported AraPheno payload structure for phenotype {phenotype_id}")

    df = pd.json_normalize(data)
    df["phenotype_id"] = phenotype_id
    df["phenotype_name"] = phenotype_name
    return df

frames = [
    normalize_values(raw_payloads[pid], pid, name)
    for name, pid in EXPECTED.items()
]

raw_long = pd.concat(frames, ignore_index=True, sort=False)
print(raw_long.shape)
raw_long.head()


In [ ]:
# Identify likely accession and phenotype-value fields without assuming one fixed API schema.
def first_existing(columns, candidates):
    lower = {c.lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    for c in columns:
        cl = c.lower()
        if any(candidate.lower() in cl for candidate in candidates):
            return c
    return None

accession_col = first_existing(
    raw_long.columns,
    ["accession_id", "accession", "accession.id", "accessionid", "id"]
)
value_col = first_existing(
    raw_long.columns,
    ["value", "phenotype_value", "phenotype", "score"]
)

print("Accession field:", accession_col)
print("Value field:", value_col)

if accession_col is None or value_col is None:
    raise KeyError("Could not identify accession/value fields. Inspect raw_long.columns before continuing.")

raw_long["accession_id"] = raw_long[accession_col].astype(str)
raw_long["value"] = pd.to_numeric(raw_long[value_col], errors="coerce")


## 4. Data-quality profile

The first public result of Case Study A is not an accuracy score. It is a transparent profile of the evidence base: how many observations exist, how many accessions are represented, how replicated each endpoint is, and where information is missing.


In [ ]:
quality = (
    raw_long.groupby(["phenotype_id", "phenotype_name"], as_index=False)
    .agg(
        n_rows=("value", "size"),
        n_nonmissing=("value", "count"),
        n_accessions=("accession_id", "nunique"),
        mean=("value", "mean"),
        std=("value", "std"),
        median=("value", "median"),
        minimum=("value", "min"),
        maximum=("value", "max"),
    )
)

quality["missing_rate"] = 1 - quality["n_nonmissing"] / quality["n_rows"]
quality


In [ ]:
replicate_profile = (
    raw_long.groupby(["phenotype_name", "accession_id"])
    .size()
    .rename("n_replicates")
    .reset_index()
)

replicate_summary = (
    replicate_profile.groupby("phenotype_name")["n_replicates"]
    .agg(["count", "min", "median", "mean", "max"])
)
replicate_summary


## 5. Visualize phenotype distributions

Regenerated-shoot counts are biological count outcomes and may be highly non-Gaussian. Distributional inspection informs the later choice between Gaussian mixed models, transformed outcomes, count-aware models, or robust alternatives.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for phenotype_name, df in raw_long.groupby("phenotype_name"):
    values = df["value"].dropna().to_numpy()
    if len(values):
        ax.hist(values, bins=30, alpha=0.35, label=phenotype_name)

ax.set_xlabel("Regenerated shoot count")
ax.set_ylabel("Number of observations")
ax.set_title("AraPheno Study 80 — focal regeneration phenotypes")
ax.legend(frameon=False)
fig.tight_layout()

fig_path = FIGURES / "case_study_a_shoot_distributions.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()


## 6. Construct accession-level phenotype summaries

The raw measurements are preserved. For the first genomic benchmark, accession-level means provide a transparent starting summary, while replicate-aware mixed modelling remains available as a sensitivity analysis.

This is not yet the final modelling dataset because genomic availability has not been intersected.


In [ ]:
accession_summary = (
    raw_long.groupby(["accession_id", "phenotype_name"], as_index=False)
    .agg(
        phenotype_mean=("value", "mean"),
        phenotype_median=("value", "median"),
        phenotype_sd=("value", "std"),
        n_replicates=("value", "count"),
    )
)

wide_mean = accession_summary.pivot(
    index="accession_id",
    columns="phenotype_name",
    values="phenotype_mean",
)

wide_mean.head()


## 7. Quantify the early-forecasting sample intersections

The core longitudinal comparison requires the same accession to have both Day-15 and Day-21 evidence under a given protocol.

For protocol $p\in\{A,B\}$, the paired set is

\[
\mathcal{A}_{p}=\mathcal{A}_{15d,p}\cap\mathcal{A}_{21d,p}.
\]

The counts below determine how much real information is available for the later comparison

\[
G + P + X_{15d} \rightarrow \widehat{Y}_{21d}
\]

versus

\[
G + P \rightarrow \widehat{Y}_{21d}.
\]


In [ ]:
pair_rows = []
for protocol in ("a", "b"):
    c15 = f"shoots 15d protocol {protocol}"
    c21 = f"shoots 21d protocol {protocol}"

    a15 = set(wide_mean.index[wide_mean[c15].notna()])
    a21 = set(wide_mean.index[wide_mean[c21].notna()])
    paired = a15 & a21

    pair_rows.append({
        "protocol": protocol.upper(),
        "n_day15": len(a15),
        "n_day21": len(a21),
        "n_paired_day15_day21": len(paired),
        "pair_retention_vs_day21": len(paired) / len(a21) if a21 else np.nan,
    })

paired_summary = pd.DataFrame(pair_rows)
paired_summary


## 8. Define the pre-genomic modelling population

The phenotype-only population is the union of accessions represented in the focal endpoints. The final modelling population is not frozen here. It will be frozen only after intersection with the 1001 Genomes accession set in `02_genomic_structure.ipynb`.

\[
\mathcal{A}_{\mathrm{model}}=\mathcal{A}_{\mathrm{phenotype}}\cap\mathcal{A}_{\mathrm{genomic}}.
\]


In [ ]:
phenotype_accessions = sorted(raw_long.loc[raw_long["value"].notna(), "accession_id"].unique())

phenotype_accession_table = pd.DataFrame({"accession_id": phenotype_accessions})
phenotype_accession_table.to_csv(
    INTERIM / "phenotype_accessions.csv",
    index=False
)

accession_summary.to_csv(
    INTERIM / "shoot_regeneration_accession_summary.csv",
    index=False
)

quality.to_csv(
    RESULTS / "case_study_a_phenotype_quality.csv",
    index=False
)

paired_summary.to_csv(
    RESULTS / "case_study_a_early_forecasting_overlap.csv",
    index=False
)

print("Phenotype accessions available before genomic intersection:", len(phenotype_accessions))


## 9. Reproducibility manifest

A small manifest records source endpoints, retrieval time, and raw-file SHA-256 checksums. This makes the public analysis auditable without requiring raw source files to remain permanently committed.


In [ ]:
manifest_rows = []
for phenotype_name, phenotype_id in EXPECTED.items():
    path = RAW / f"phenotype_{phenotype_id}.json"
    sha256 = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest_rows.append({
        "study_id": STUDY_ID,
        "phenotype_id": phenotype_id,
        "phenotype_name": phenotype_name,
        "source_url": f"{ARAPHENO}/phenotype/{phenotype_id}/values.json",
        "retrieved_at_utc": retrieved_at,
        "sha256": sha256,
    })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(RAW / "manifest.csv", index=False)
manifest


## Decision for the next notebook

`02_genomic_structure.ipynb` should now:

1. retrieve or reference the compatible 1001 Genomes accession/genotype resource;
2. intersect genomic accessions with `phenotype_accessions.csv`;
3. quantify the final $n$ and genomic dimensionality $p$;
4. examine missingness and marker filtering;
5. characterize genomic relatedness and population structure;
6. design genotype-aware validation folds;
7. freeze the exact modelling population that will feed the classical quantitative-genetics baseline in `03_genomic_prediction.ipynb`.

Only after that intersection is established should Model 1 be fitted.
